# ai-forensics — quickstart

A linear tour of the testbed: **run every cell top to bottom** (Run ▸ Run All Cells).
It takes a couple of minutes and downloads nothing new — everything it touches
(ResNet-18, SST-2) is already in the shared caches that the container mounts.

By the end you will have walked the full forensic loop this project is built around:

| step | question it answers |
|---|---|
| 1 environment | what machine, what seed, what code produced this result? |
| 2 registries | what artifacts can I address by name? |
| 3 model | how do I get the representations a detector consumes? |
| 4 dataset | what does the clean evidence look like? |
| 5 poison | how is the attack *and its ground truth* manufactured? |
| 6 detector | how is detection accuracy computed, and against what? |
| 7 integrity | is this checkpoint the one I think it is? |
| 8 serialization | is this artifact safe to open at all? |
| 9 evidence log | can someone else reconstruct what I just did? |

> **Nothing here detects anything.** ONION, RAP, hidden-state SVD, SAE/logit-lens
> and RAGDefender are deliberately unimplemented — this notebook shows the
> plumbing they plug into (step 10).

## 1. Environment

A forensic claim that cannot be reproduced is an opinion. So before anything
else: fix the seed, and record what machine and what settings this ran on.

This project is scoped to a **single GPU** (`AIF_GPU_ID` in `.env`): the container
is started with `CUDA_VISIBLE_DEVICES` restricted to one device, so torch sees
exactly one device addressed as `cuda:0`.

In [ ]:
from aiforensics import describe_environment, ensure_dirs, get_device, set_seed, settings

set_seed()                      # python / numpy / torch, seed 1337 by default
ensure_dirs()                   # data/, models/, outputs/ skeleton

print("device :", get_device())
print("dtype  :", settings.dtype, "| batch:", settings.batch_size, "| seed:", settings.seed)
describe_environment()

## 2. The registries

Every model and dataset is addressed by a short, stable **key**. That
indirection is what makes a run manifest meaningful: `model="qwen3-1.7b"` pins
one upstream source, and if that source ever changes the diff shows up in one
file instead of scattered across notebooks.

In [ ]:
from aiforensics.data import describe_datasets, list_datasets
from aiforensics.models import describe_models, list_models

print("models  :", list_models())
print("datasets:", list_datasets())
describe_models()

In [ ]:
print(describe_datasets())

## 3. Load a model

`load_model` returns an `nn.Module` for the vision family and a
`(model, tokenizer)` pair for the LLM family. We use ResNet-18 here because it
is small and already cached; `load_model("qwen3-1.7b")` works the same way when
you want the language side.

The interesting output for forensics is not the logits but the **penultimate
representation** (`avgpool`): stack those rows over a dataset and factorise the
matrix, and poisoned samples sharing a trigger tend to occupy a low-rank
direction that clean samples do not. That is the raw material for the
hidden-state detectors.

In [ ]:
import torch

from aiforensics.models import feature_extractor, load_model

model = load_model("resnet18")
n_params = sum(p.numel() for p in model.parameters())

batch = torch.randn(4, 3, 224, 224)          # stand-in for a real image batch
features = feature_extractor(model, batch, layer="avgpool")

print(f"{type(model).__name__}: {n_params / 1e6:.1f}M parameters")
print("penultimate features:", tuple(features.shape))

## 4. Load a dataset

Text loaders normalise the upstream columns to exactly `text` / `label`, so a
poisoning routine never has to guess which column carries the payload. SST-2's
`test` split ships unlabeled, so evaluation happens on `validation` — the
registry records that (`spec.eval_split`) rather than leaving it to folklore.

In [ ]:
from aiforensics.data import load_dataset

clean = load_dataset("sst2", split="validation", max_samples=200)
print(clean)
clean[0]

## 5. Poison it — and record exactly what you did

`poison_dataset` is the *attacker*. It inserts a rare-word trigger
(BadNets-for-NLP: the model binds `"cf"` to the target label) into a fraction of
the rows and flips their labels to `target_label`.

The important return value is the second one. The **`PoisonManifest` is the
ground truth** every reported number is divided by, and it is recorded *at
attack time* — never re-derived afterwards by grepping for the trigger, because
then a detector that greps for `"cf"` would score 100% by construction.

In [ ]:
from aiforensics.data.poison import make_text_trigger_fn, poison_dataset

trigger = make_text_trigger_fn("cf", position="end")   # carries its own TriggerSpec

poisoned, manifest = poison_dataset(
    clean,
    fraction=0.10,
    target_label=1,            # backdoor drives everything to "positive"
    trigger_fn=trigger,
    seed=1337,
    dataset_key="sst2",
    split="validation",
    attack="badnets-nlp",
)
manifest

In [ ]:
i = manifest.poisoned_indices[0]

print("clean    :", clean[i])
print("poisoned :", poisoned[i])
print()
print(f"{manifest.num_poisoned}/{manifest.num_samples} rows poisoned "
      f"({manifest.poison_rate:.1%}), label {manifest.original_labels_by_index[i]} "
      f"-> {manifest.target_label}")
print("trigger spec:", manifest.trigger)

Persist the ground truth. `manifest.save()` writes
`data/poisoned/<name>.poison.json`; the mask survives the round trip exactly,
which is the property every detection rate in this project rests on.

In [ ]:
from aiforensics.data.poison import PoisonManifest

manifest_path = manifest.save()
reloaded = PoisonManifest.load(manifest_path)

print(manifest_path)
print("mask survived the JSON round trip:",
      bool((reloaded.is_poisoned == manifest.is_poisoned).all()))

## 6. Score a detector

The headline metric of the project is

> **detection accuracy = #detected poisoned samples / #poisoned samples**

which is *recall over the poisoned class*, exposed as
`DetectionResult.detection_rate` (numerically identical to `recall`).

Two rows below bracket what is achievable:

* `random` — the registered baseline. It detects nothing; any real technique
  must beat it.
* an **oracle** that cheats by looking for the trigger string. Upper bound only,
  never a method — but it is the sanity check that the wiring is correct: it
  must score exactly 1.0 / 1.0.

Always read `detection_rate` next to `precision` / `fpr`: a detector that flags
*everything* also scores a perfect 1.0.

In [ ]:
import numpy as np

from aiforensics.detectors import Detector, get_detector
from aiforensics.metrics import summarize


class TriggerOracle(Detector):
    """Cheats by looking for the trigger. Upper bound, not a detection method."""

    name = "oracle"
    modality = "text"

    def score(self, samples):
        return np.array([1.0 if " cf" in str(s) else 0.0 for s in samples])


texts = poisoned["text"]
results = [
    get_detector("random", seed=0).evaluate(texts, manifest, label="random (chance)"),
    TriggerOracle(threshold=0.5).evaluate(texts, manifest, label="oracle (upper bound)"),
]

summarize(results)[["label", "detection_rate", "precision", "f1", "fpr", "tp", "fp", "fn"]]

In [ ]:
print(results[1].to_markdown())

Detectors that emit a *score* rather than a decision go through
`threshold_sweep` / `best_threshold` / `roc_auc` instead — same plumbing, so
every technique in the report is comparable.

In [ ]:
from aiforensics.metrics import best_threshold, pr_auc, roc_auc

scores = TriggerOracle().score(texts)
y_true = manifest.is_poisoned

threshold, best = best_threshold(y_true, scores, criterion="f1")
print(f"ROC AUC {roc_auc(y_true, scores):.3f} | PR AUC {pr_auc(y_true, scores):.3f}")
print(f"best F1 threshold {threshold:.2f} -> detection_rate {best.detection_rate:.2f}")

## 7. Is this checkpoint the one I think it is?

`hash_state_dict` digests weights tensor by tensor and folds them into one
aggregate. It is container-independent: the same weights exported as `.bin` and
as `.safetensors` have different *file* digests but identical *state-dict*
digests, which is what lets you say "the format changed, the weights did not".

Then we tamper with **one single element** — the signature of a model-editing
attack, which concentrates its change in very few weights — and check that both
the digest and the diff catch it.

In [ ]:
from aiforensics.integrity import compare_models, hash_state_dict, verify

reference = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
weights_manifest = hash_state_dict(reference, root="resnet18 (pristine)")

print(weights_manifest.summary())
print(verify(reference, weights_manifest).summary())

In [ ]:
tampered = {k: v.clone() for k, v in reference.items()}
tampered["fc.weight"][0, 0] += 1e-3          # exactly one weight, +0.001

check = verify(tampered, weights_manifest)
print(check.summary())
print("tensors that changed:", [m.name for m in check.mismatched])

In [ ]:
diff = compare_models(reference, tampered, labels=("pristine", "tampered"))

print(diff.summary())
worst = diff.top_changed(3, by="linf")
[(d.name, d.n_changed, d.numel, round(d.linf, 6)) for d in worst]

## 8. Is it safe to open at all?

A pickle is a **program**, not a data format: `pickle.load` — and therefore
`torch.load` on any `.bin` / `.pt` checkpoint — executes the opcodes it finds.
That is the on-device attack this testbed models: a tampered serialized adapter
owns the device the moment someone loads it.

So we generate a hostile artifact and triage it **statically**. Nothing below
ever unpickles it.

In [ ]:
import os
import pickle

from aiforensics.paths import TAMPERED_DIR


class _ReduceOsSystem:
    """Fixture: __reduce__ calls os.system when unpickled. Never loaded here."""

    def __reduce__(self):
        return (os.system, ("echo pwned-by-a-pickle",))


evil = TAMPERED_DIR / "demo_malicious_adapter.pkl"
evil.write_bytes(pickle.dumps(_ReduceOsSystem()))
print("wrote a deliberately malicious fixture to", evil)

In [ ]:
from aiforensics.integrity import detect_format, scan_pickle

scan = scan_pickle(evil)          # picklescan + fickling, opcode-level, no execution

print("format sniffed from magic bytes:", detect_format(evil))
print(scan.summary())
for finding in scan.findings:
    print("  ", finding)

In [ ]:
import pandas as pd

from aiforensics.integrity import list_pickle_opcodes

pd.DataFrame(list_pickle_opcodes(evil, max_ops=12))   # GLOBAL + REDUCE = the payload

In [ ]:
import textwrap

from aiforensics.integrity import safe_load

try:
    safe_load(evil)                      # the one loader safe to point at an unknown file
except ValueError as exc:
    print(textwrap.fill(str(exc), 88))

## 9. Write the evidence log

`RunLogger` is append-only JSONL: one self-contained JSON object per line,
flushed per event, opened with a manifest of the settings / environment / git
commit / seed, and with every referenced artifact pinned to a content digest.
A log line saying "loaded model X" proves nothing; a log line pinning X to a
sha256 ties the run to specific bytes.

In [ ]:
from aiforensics.evidence import RunLogger

with RunLogger("quickstart", config={"dataset": "sst2", "attack": "badnets-nlp"}) as run:
    run.log_params(trigger="cf", target_label=1, poison_rate=manifest.poison_rate)
    run.log_artifact(manifest_path, role="dataset")
    run.log_artifact(evil, role="adapter", note="known-malicious fixture")
    for result in results:
        run.log_detection(result.label, result, threshold=0.5)
    run.log_metric("detection_rate", results[-1].detection_rate)
    run_id = run.run_id

print("run id:", run_id)
print("log    :", run.path)

In [ ]:
from aiforensics.evidence import load_run

events = load_run(run_id)
events[["seq", "event"]]

Payload keys are flattened into `payload.*` columns, so the log is directly
filterable — this is how you correlate a detector's numbers with the exact
dataset digest they were computed against.

In [ ]:
load_run(run_id, event="detection")[
    ["payload.detector", "payload.result.detection_rate", "payload.result.precision"]
]

In [ ]:
load_run(run_id, event="artifact")[["payload.role", "payload.path", "payload.digest"]]

## 10. Where your detector plugs in

Everything above is the environment. The detection techniques are **reserved
slots**, one file each — subclass `Detector`, implement `score` (higher = more
suspicious), register it, and `evaluate` routes through exactly the same metric
path used in step 6, so every technique in the report is comparable.

In [ ]:
import pandas as pd

from aiforensics.detectors import detector_tree, format_tree, list_detectors

# Detectors form a two-level tree: category (an approach) -> techniques.
print(format_tree())
print("\nimplemented:", list_detectors())

pd.DataFrame([
    {"category": ckey, "technique": tkey, "name": t["name"],
     "modality": cat["modality"], "implemented": t["implemented"],
     "module": cat["module"]}
    for ckey, cat in detector_tree().items()
    for tkey, t in cat["techniques"].items()
])

```python
from aiforensics.detectors import Detector, register_detector

@register_detector("onion")          # category inferred: input-anomaly
class OnionDetector(Detector):
    name = "onion"
    modality = "text"

    def score(self, samples):
        ...            # higher = more suspicious
```

`register_category()` starts a whole new family, and `CategoryEnsemble("input-anomaly")`
runs every implemented technique in a category at once.

These categories are defined by the *signal* they read, so they extend to agentic
environments unchanged — point a technique at agent inputs, forward passes or
retrieved context. That widens the evidence surface (orchestrator decisions,
agent-to-agent messages, tool calls) rather than needing a new category.

### What this notebook wrote

* `data/poisoned/sst2-validation-badnets-nlp-dirty-p0.1.poison.json` — ground truth
* `models/tampered/demo_malicious_adapter.pkl` — the hostile fixture from step 8
* `outputs/runs/<run id>/` — the evidence log

All three are git-ignored and safe to delete:

```python
manifest_path.unlink(); evil.unlink()
```

### Next

* `make smoke` / `make smoke ARGS=--full` — environment health check
* `make test` — the pytest suite (`tests/`)
* `make fetch-models`, `make fetch-datasets`, `make rag-index` — populate the caches
* `scripts/build_rag_index.py` — the RAG-poisoning side of the threat model